In [ ]:
#imports
import torch, pickle, copy, torchvision, torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torch.optim import lr_scheduler
from pytorch_msssim import MS_SSIM
import matplotlib.pyplot as plt

In [ ]:
#encoder architecture

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.Flt = nn.Flatten()
        
        #branch 1
        self.L1_1 = nn.Linear(784, 512)
        self.L2_1 = nn.Linear(512, 128)

        #brach 2
        self.L1_2 = nn.Linear(784, 256)
        self.L2_2 = nn.Linear(256, 64)

        self.L1 = nn.Linear(192, 64)

        self.gelu = nn.GELU()
    
    def forward(self, img):
        
        out = self.Flt(img)
        
        #branch 1
        out1 = out
        out1 = self.gelu(self.L1_1(out1))
        out1 = self.gelu(self.L2_1(out1))

        #branch 2
        out2 = out
        out2 = self.gelu(self.L1_2(out2))
        out2 = self.gelu(self.L2_2(out2))

        #concat
        out = torch.cat((out1, out2), dim=1)
        
        out = self.gelu(self.L1(out))

        return out

In [ ]:
#decoder architecture

class Decoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.Exp = nn.Linear(64, 192)
        
        #branch 1
        self.L1_1 = nn.Linear(192, 64)
        self.L2_1 = nn.Linear(64, 256)

        #brach 2
        self.L1_2 = nn.Linear(192, 128)
        self.L2_2 = nn.Linear(128, 512)

        self.L1 = nn.Linear(768, 784)

        self.gelu = nn.GELU()
        self.sigm = nn.Sigmoid()
    
    def forward(self, embedding):
        
        out = self.Exp(embedding)
        
        #branch 1
        out1 = out
        out1 = self.gelu(self.L1_1(out1))
        out1 = self.gelu(self.L2_1(out1))

        #branch 2
        out2 = out
        out2 = self.gelu(self.L1_2(out2))
        out2 = self.gelu(self.L2_2(out2))

        #concat
        out = torch.cat((out1, out2), dim=1)
        
        out = self.sigm(self.L1(out))

        return out.view(-1, 1, 28, 28)

In [ ]:
#autoencoder architecture

class Autoencoder(nn.Module):
    
    def __init__(self):

        super().__init__()

        self.Enc = Encoder()
        self.Dec = Decoder()

    def forward(self, img):

        out = self.Enc(img)
        out = self.Dec(out)

        return out

In [ ]:
#data preprocessing

batch_size = 64

train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation((-7, +7)),
])

val_test_transforms = transforms.Compose([
    transforms.ToTensor()
])

#train

gen1 = torch.Generator().manual_seed(13)
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True, 
    transform=train_transforms, 
    download=True
)
train_subset, __ = random_split(
    dataset=train_dataset, 
    lengths=[54000, 6000], 
    generator=gen1
)
train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=True)

#val

gen2 = torch.Generator().manual_seed(13)
val_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True, 
    transform=val_test_transforms, 
    download=True
)
__, val_subset = random_split(
    dataset=val_dataset, 
    lengths=[54000, 6000], 
    generator=gen2
)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

#test

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=False, 
    transform=val_test_transforms, 
    download=True
)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
#defining the loss function class

class StructuralLoss(nn.Module):
    def __init__(self):

        super().__init__()
        self.ms_ssim = MS_SSIM(
            data_range=1.0,
            size_average=True,
            win_size=7,
            channel=1,
            weights=[0.333, 0.333, 0.333] 
            #to limit the number of levels in the downsampling pyramid
        )
    def forward(self, out_imgs, in_imgs):
        return 1-self.ms_ssim(out_imgs, in_imgs)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_model(model, optimizer, criterion, train_loader, val_loader, scheduler, loss_dict, acc_dict,  n_epochs=100):

    best_MSSSIM = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(n_epochs):

        #train

        model.train()

        running_MSSSIM_sum = 0.0
        length = 0

        for i, (in_imgs, _) in enumerate(train_loader):
            
            in_imgs = in_imgs.to(device)
            out_imgs = model(in_imgs)
            
            loss = criterion(out_imgs, in_imgs)
            loss.backward()

            with torch.no_grad():
                running_MSSSIM_sum += (1-loss.item())*(in_imgs.size(0))
                length += in_imgs.size(0)

            optimizer.step()
            optimizer.zero_grad()
        
        epoch_MSSSIM = running_MSSSIM_sum/length
        epoch_loss = 1-epoch_MSSSIM
        loss_dict['train'].append(epoch_loss)
        acc_dict['train'].append(epoch_MSSSIM)

        #val

        model.eval()

        with torch.no_grad():

            running_MSSSIM_sum = 0.0
            length = 0

            for i, (in_imgs, _) in enumerate(val_loader):
                
                in_imgs = in_imgs.to(device)
                out_imgs = model(in_imgs)
                
                loss = criterion(out_imgs, in_imgs)
                
                running_MSSSIM_sum += (1-loss.item())*(in_imgs.size(0))
                length += in_imgs.size(0)
            
            val_MSSSIM = running_MSSSIM_sum/length
            val_loss = 1-val_MSSSIM

            loss_dict['val'].append(val_loss)
            acc_dict['val'].append(val_MSSSIM)

            if (val_MSSSIM > best_MSSSIM):
                best_MSSSIM = val_MSSSIM
                best_wts = copy.deepcopy(model.state_dict())
        
        scheduler.step()

        print('-'*100)
        print(f'epoch {epoch+1}')
        print(f'train_MSSSIM:{epoch_MSSSIM}')
        print(f'val_MSSSIM  :{val_MSSSIM}')

    with open('acc_loss.bin', 'wb') as file:
        pickle.dump([loss_dict, acc_dict], file)

    torch.save(best_wts, 'autoencoder_wts.pth')

    model.load_state_dict(best_wts)

    return model

In [ ]:
model = Autoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = StructuralLoss()
scheduler = lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
loss_dict = {}
acc_dict = {}


train_model(model, optimizer, criterion, train_loader, val_loader, scheduler, loss_dict, acc_dict,  n_epochs=100)


In [ ]:
with open('acc_loss.bin', 'rb') as file:
    l = pickle.load(file)

train_loss = l[0]['train']
val_loss = l[0]['val']
train_acc = l[1]['train']
val_acc = l[1]['val']
epochs = [i+1 for i in range(100)]

plt.figure()
plt.plot(epochs, train_loss, label='train_loss')
plt.plot(epochs, val_loss, label='val_loss')
plt.legend()

plt.figure()
plt.plot(epochs, train_acc, label='train_MSSSIM')
plt.plot(epochs, val_acc, label='val_MSSSIM')
plt.legend()